# MLP Memory Analysis

Analyze MLPs as key-value memories: activation patterns, retrieval behavior,
storage capacity, input selectivity, and write-read alignment.

## Why This Matters

MLP memory provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_memory_analysis import (
    mlp_key_value_decomposition, mlp_retrieval_pattern,
    mlp_storage_capacity, mlp_input_selectivity,
    mlp_write_read_alignment,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Key-Value Decomposition

How many neurons activate at each position?

In [ ]:
result = mlp_key_value_decomposition(model, tokens, layer=0)
print(f"Neurons: {result['n_neurons']}, mean active fraction: {result['mean_active_fraction']:.2%}\n")
for p in result['per_position']:
    bar = '█' * int(p['active_fraction'] * 40)
    print(f"  Pos {p['position']}: {p['n_active_neurons']:3d} active "
          f"({p['active_fraction']:.1%}), mean_act={p['mean_activation']:.4f} {bar}")

## Retrieval Pattern

What tokens does the MLP promote/suppress at each position?

In [ ]:
result = mlp_retrieval_pattern(model, tokens, layer=0, top_k=5)
print(f"Mean output norm: {result['mean_output_norm']:.4f}\n")
for p in result['per_position']:
    print(f"  Pos {p['position']}: norm={p['output_norm']:.4f}, "
          f"promotes={p['top_promoted'][:3]}, "
          f"max_logit={p['max_logit']:+.4f}")

## Storage Capacity

How much of the MLP's capacity is being used?

In [ ]:
result = mlp_storage_capacity(model, tokens, layer=0)
print(f"Total neurons: {result['n_neurons']}")
print(f"Ever active: {result['n_ever_active']} ({result['utilization']:.1%})")
print(f"Effective rank: {result['effective_rank']:.2f}")
print(f"Mean neuron reuse: {result['mean_neuron_reuse']:.2f} positions")

## Input Selectivity

Which neurons are most position-selective?

In [ ]:
result = mlp_input_selectivity(model, tokens, layer=0, top_k=10)
print(f"Selective neurons: {result['n_selective']}")
print(f"Mean selectivity: {result['mean_selectivity']:.4f}\n")
for n in result['per_neuron']:
    print(f"  Neuron {n['neuron_idx']:3d}: sel={n['selectivity']:.3f}, "
          f"peak_pos={n['peak_position']}, "
          f"peak_act={n['peak_activation']:.4f}")

## Write-Read Alignment

Do later layers read what this MLP writes?

In [ ]:
result = mlp_write_read_alignment(model, tokens, layer=0)
print(f"Source: layer {result['source_layer']}")
print(f"Reading layers: {result['n_reading_layers']}")
print(f"Mean downstream alignment: {result['mean_downstream_alignment']:.4f}\n")
for p in result['per_target']:
    read = 'READS' if p['is_read'] else 'ignores'
    print(f"  → Layer {p['target_layer']}: alignment={p['mean_alignment']:+.4f} [{read}]")